
# AhorraMás Web Scraper

This notebook collects product and price information from AhorraMás using Selenium and BeautifulSoup.

### How it works

1. Opens the AhorraMás website and accepts cookies.
2. Searches for the product terms defined in `SEARCH_TERMS`.
3. Loads additional products using the "Más resultados" button.
4. Extracts product names, IDs, prices, units, URLs, and timestamps.
5. Reviews the collected RAW data and reports incomplete searches.
6. Saves the current RAW collection and a dated historical RAW snapshot.

### Data storage

- Current RAW: `data/raw/ahorramas_products_raw.csv`
- Historical RAW: `data/history/ahorramas/YYYY-MM-DD_raw.csv`

Deduplication, standardization, classification, and historical price processing are handled by the central pipeline.

TEST_MODE allows extraction tests without saving files.



In [ ]:

import time
import re
from pathlib import Path
from datetime import datetime
from urllib.parse import urljoin, quote

import pandas as pd
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException


HOME_URL = "https://www.ahorramas.com/"

SEARCH_TERMS = [
    "leche",
    "huevos",
    "manzana",
    "plátano",
    "tomate",
    "arroz",
    "pasta",
    "pan",
    "pollo",
    "carne picada",
    "salmón",
    "yogur",
    "queso",
    "agua",
    "zumo",
    "atún",
    "garbanzos",
    "aceite",
    "detergente",
    "papel higiénico",
]

# --------------------------------------------------
# TEST MODE
# --------------------------------------------------

TEST_MODE = False
TEST_TERMS = ["leche"]

TERMS_TO_SCRAPE = TEST_TERMS if TEST_MODE else SEARCH_TERMS

print("Search terms to scrape:", TERMS_TO_SCRAPE)



# --------------------------------------------------
# PROJECT DIRECTORIES
# --------------------------------------------------

def find_project_root():
    current = Path.cwd().resolve()

    for path in (current, *current.parents):
        if path.name == "supermarket_project":
            return path

    if current.name == "scrapers":
        return current.parent

    raise FileNotFoundError("Could not locate the project root.")


BASE_DIR = find_project_root()

RAW_DIR = BASE_DIR / "data" / "raw"
HISTORY_DIR = BASE_DIR / "data" / "history" / "ahorramas"

RAW_PATH = RAW_DIR / "ahorramas_products_raw.csv"

RUN_DATE = datetime.now().strftime("%Y-%m-%d")
RAW_HISTORY_PATH = HISTORY_DIR / f"{RUN_DATE}_raw.csv"

print("Project:", BASE_DIR)
print("Collection date:", RUN_DATE)
print("Current RAW:", RAW_PATH)
print("Historical RAW:", RAW_HISTORY_PATH)


# --------------------------------------------------
# SELENIUM DRIVER CONFIGURATION
# --------------------------------------------------

def start_ahorramas_driver():
    options = Options()

    options.add_argument("--window-size=1920,1080")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option(
        "excludeSwitches", ["enable-automation"]
    )
    options.add_experimental_option(
        "useAutomationExtension", False
    )
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/122.0.0.0 Safari/537.36"
    )

    driver = webdriver.Chrome(options=options)

    driver.execute_script(
        "Object.defineProperty(navigator, 'webdriver', "
        "{get: () => undefined})"
    )

    driver.get(HOME_URL)

    return driver


def accept_ahorramas_cookies(driver):
    xpaths = [
        "//button[contains(., 'Aceptar')]",
        "//button[contains(., 'Permitir')]",
        "//button[@id='onetrust-accept-btn-handler']"
    ]

    for xpath in xpaths:
        try:
            buttons = driver.find_elements(By.XPATH, xpath)

            for btn in buttons:
                if btn.is_displayed():
                    btn.click()
                    print("AhorraMás cookies accepted.")
                    return

        except Exception:
            pass

    print("Cookie banner not found or already accepted.")


# --------------------------------------------------
# PRODUCT SEARCH AND PAGINATION
# --------------------------------------------------

def count_product_links(driver):
    # Count unique links matching potential product URLs
    links = driver.find_elements(
        By.CSS_SELECTOR,
        'a[href*=".html"], a[class*="product"], div.product-tile a'
    )

    hrefs = {
        link.get_attribute("href")
        for link in links
        if link.get_attribute("href")
    }

    return len(hrefs)


def search_ahorramas(driver, term):
    encoded_term = quote(term)
    search_url = f"https://www.ahorramas.com/search?q={encoded_term}"

    print(f"Navigating to: {search_url}")

    driver.get(search_url)
    time.sleep(4)


def load_all_ahorramas_products(driver, max_clicks=100):

    print("Loading all available products...")

    clicks = 0
    selector = "button.more[data-url]"

    while clicks < max_clicks:

        # Find the "More results" button in the DOM
        buttons = driver.find_elements(By.CSS_SELECTOR, selector)

        enabled_buttons = [
            button for button in buttons
            if button.is_displayed() and button.is_enabled()
        ]

        if not enabled_buttons:

            # Distinguish hidden buttons from completed pagination
            if buttons:
                print("More results button exists but is not visible.")
                return count_product_links(driver), "button_not_visible"

            print("No more results button found.")
            return count_product_links(driver), "complete"

        button = enabled_buttons[0]

        # Scroll directly to the button instead of the page bottom
        driver.execute_script(
            "arguments[0].scrollIntoView({"
            "behavior: 'instant', block: 'center'"
            "});",
            button
        )

        time.sleep(0.5)

        previous_count = count_product_links(driver)
        previous_url = button.get_attribute("data-url")

        try:
            # Load the next batch of products
            button.click()

            # Wait until new products or updated pagination are detected
            WebDriverWait(driver, 20).until(
                lambda d: (
                    count_product_links(d) > previous_count
                    or not any(
                        b.is_displayed()
                        and b.get_attribute("data-url") == previous_url
                        for b in d.find_elements(
                            By.CSS_SELECTOR, selector
                        )
                    )
                )
            )

        except TimeoutException:
            print("Timeout loading additional products.")
            return count_product_links(driver), "timeout"

        except Exception as exc:
            print(f"Error: {type(exc).__name__}: {exc}")
            return count_product_links(driver), "error"

        clicks += 1

        current_count = count_product_links(driver)

        print(
            f"Click {clicks}: "
            f"{previous_count} -> {current_count} links"
        )

    print("Maximum number of clicks reached.")
    return count_product_links(driver), "max_clicks_reached"


# --------------------------------------------------
# PRODUCT DATA EXTRACTION
# --------------------------------------------------

def extract_ahorramas_products(driver, searched_item):

    soup = BeautifulSoup(driver.page_source, "html.parser")

    cards = soup.select(
        'div.product-tile, li.grid-tile, div.grid-tile, '
        'div[class*="product-tile"]'
    )

    if not cards:
        cards = soup.select('div[class*="product"], article')

    records = []
    seen_urls = set()

    scraped_at = datetime.now().isoformat(timespec="seconds")

    for card in cards:
        text = card.get_text(" ", strip=True)

        # Identify the product URL
        link_elem = card.find(
            "a", href=re.compile(r"\.html|/p/")
        )

        if not link_elem:
            link_elem = card.find("a", href=True)

        if not link_elem:
            continue

        href = link_elem.get("href")

        if not href or "search" in href or "cart" in href:
            continue

        url = urljoin(HOME_URL, href)

        if url in seen_urls:
            continue

        seen_urls.add(url)

        # Extract product name
        name = ""

        for tag in ["h3", "h2", "h4", "a", "span", "div"]:
            elem = card.find(
                tag,
                class_=re.compile("title|name|desc|link", re.I)
            )

            if elem:
                txt = elem.get_text(" ", strip=True)

                if len(txt) > 4 and "€" not in txt:
                    name = txt
                    break

        if not name:
            link_text = link_elem.get_text(" ", strip=True)

            if len(link_text) > 4:
                name = link_text

        if not name:
            continue

        # Extract total product price
        price = None
        price_raw = None

        price_match = re.search(
            r"(\d+(?:[.,]\d{1,2})?)\s*€",
            text
        )

        if price_match:
            price_raw = price_match.group(0)
            price = float(
                price_match.group(1).replace(",", ".")
            )

        # Discard only missing, zero or negative prices
        if price is None or price <= 0:
            continue

        # Extract product ID from URL
        id_match = re.search(
            r"/(\d+)\.html|/p/(\d+)", url
        )

        product_id = (
            id_match.group(1) or id_match.group(2)
            if id_match else None
        )

        # --------------------------------------------------
        # EXTRACT QUANTITY AND UNIT PRICE
        # --------------------------------------------------

        unit = None
        price_per_unit = None
        quantity_val = None

        name_lower = name.lower()

        # Detect liters and milliliters
        l_match = re.search(
            r"(\d+(?:[.,]\d+)?)\s*(?:l|litros?)\b",
            name_lower
        )

        ml_match = re.search(
            r"(\d+)\s*ml\b",
            name_lower
        )

        # Detect kilograms and grams
        kg_match = re.search(
            r"(\d+(?:[.,]\d+)?)\s*(?:kg|k)\b",
            name_lower
        )

        g_match = re.search(
            r"(\d+)\s*(?:g|gr)\b",
            name_lower
        )

        # Detect individual units
        u_match = re.search(
            r"(\d+)\s*(?:u|uds|unidades?)\b",
            name_lower
        )

        if l_match:
            quantity_val = float(
                l_match.group(1).replace(",", ".")
            )
            unit = "LITRO"

        elif ml_match:
            quantity_val = float(ml_match.group(1)) / 1000.0
            unit = "LITRO"

        elif kg_match:
            quantity_val = float(
                kg_match.group(1).replace(",", ".")
            )
            unit = "KILO"

        elif g_match:
            quantity_val = float(g_match.group(1)) / 1000.0
            unit = "KILO"

        elif u_match:
            quantity_val = float(u_match.group(1))
            unit = "UNIDAD"

            if quantity_val == 12 and "docena" in name_lower:
                unit = "DOCENA"

        # Calculate price per unit when quantity is available
        if quantity_val and quantity_val > 0:
            price_per_unit = round(
                price / quantity_val, 4
            )

        # Store extracted product information
        records.append({
            "searched_item": searched_item,
            "supermarket": "AhorraMás",
            "product_id": product_id,
            "name": name,
            "quantity_raw": None,
            "price_raw": price_raw,
            "price": price,
            "price_per_unit_raw": None,
            "price_per_unit": price_per_unit,
            "unit": unit,
            "url": url,
            "scraped_at": scraped_at,
        })

    df = pd.DataFrame(records)

    print(
        f"Products extracted for '{searched_item}': {len(df)}"
    )

    return df


Search terms to scrape: ['leche']
Project: C:\Users\Giuliana Galassi\OneDrive\Documentos\uc3m\4TO\WEB ANALYTICS\supermarket_project
Collection date: 2026-11-13
Current RAW: C:\Users\Giuliana Galassi\OneDrive\Documentos\uc3m\4TO\WEB ANALYTICS\supermarket_project\data\raw\ahorramas_products_raw.csv
Historical RAW: C:\Users\Giuliana Galassi\OneDrive\Documentos\uc3m\4TO\WEB ANALYTICS\supermarket_project\data\history\ahorramas\2026-11-13_raw.csv


In [2]:
all_results = []
summary = []

driver = start_ahorramas_driver()

try:
    time.sleep(3)
    accept_ahorramas_cookies(driver)

    for i, term in enumerate(TERMS_TO_SCRAPE, start=1):
        print("\n" + "=" * 70)
        print(f"[{i}/{len(TERMS_TO_SCRAPE)}] Buscando en AhorraMás: {term}")

        start = time.monotonic()

        try:
            search_ahorramas(driver, term)
            loaded_count, load_status = load_all_ahorramas_products(driver)
            df_term = extract_ahorramas_products(driver, term)

            elapsed = round(time.monotonic() - start, 2)

            if not df_term.empty:
                all_results.append(df_term)

            summary.append({
                "searched_item": term,
                "status": load_status,
                "products_loaded": loaded_count,
                "rows_extracted": len(df_term),
                "seconds": elapsed,
            })

            print(f"✓ {len(df_term)} filas extraídas en {elapsed} s")
            time.sleep(2)

        except Exception as exc:
            elapsed = round(time.monotonic() - start, 2)
            print(f"ERROR con '{term}': {type(exc).__name__}: {exc}")
            summary.append({
                "searched_item": term,
                "status": "error",
                "products_loaded": None,
                "rows_extracted": 0,
                "seconds": elapsed,
            })
            break

finally:
    driver.quit()

df_ahorramas = pd.concat(all_results, ignore_index=True) if all_results else pd.DataFrame()
df_summary = pd.DataFrame(summary)
display(df_summary)
print("\nFilas raw totales:", len(df_ahorramas))

AhorraMás cookies accepted.

[1/1] Buscando en AhorraMás: leche
Navigating to: https://www.ahorramas.com/search?q=leche
Loading all available products...
Click 1: 44 -> 64 links
Click 2: 64 -> 84 links
Click 3: 84 -> 104 links
Click 4: 104 -> 124 links
Click 5: 124 -> 144 links
Click 6: 144 -> 164 links
Click 7: 164 -> 184 links
Click 8: 184 -> 195 links
No more results button found.
Products extracted for 'leche': 167
✓ 167 filas extraídas en 202.06 s


,searched_item,status,products_loaded,rows_extracted,seconds
0,leche,complete,195,167,202.06



Filas raw totales: 167


In [3]:
# --------------------------------------------------
# PREPARE RAW DATA
# --------------------------------------------------

if df_ahorramas.empty:
    print("No raw data collected.")
else:
    print("Raw data ready for processing.")
    print("Total rows:", len(df_ahorramas))
    print("Search terms:", df_ahorramas["searched_item"].nunique())


Raw data ready for processing.
Total rows: 167
Search terms: 1


In [4]:

# --------------------------------------------------
# VALIDATE AND SAVE RAW DATA
# --------------------------------------------------

import os
import tempfile

if TEST_MODE:
    print("TEST MODE: no files will be saved.")

else:
    # --------------------------------------------------
    # VALIDATE COLLECTED DATA
    # --------------------------------------------------

    if df_ahorramas.empty:
        raise RuntimeError(
            "No products collected. No files saved."
        )

    required_columns = [
        "searched_item",
        "supermarket",
        "product_id",
        "name",
        "quantity_raw",
        "price_raw",
        "price",
        "price_per_unit_raw",
        "price_per_unit",
        "unit",
        "url",
        "scraped_at",
    ]

    missing_columns = [
        col for col in required_columns
        if col not in df_ahorramas.columns
    ]

    if missing_columns:
        raise RuntimeError(
            f"Missing RAW columns: {missing_columns}"
        )

    if df_summary.empty:
        raise RuntimeError(
            "No search results were recorded. No files saved."
        )

    valid_names = (
        df_ahorramas["name"].notna()
        & df_ahorramas["name"].astype(str).str.strip().ne("")
    )

    numeric_prices = pd.to_numeric(
        df_ahorramas["price"], errors="coerce"
    )

    valid_prices = numeric_prices.notna() & numeric_prices.gt(0)

    if not valid_names.any() or not valid_prices.any():
        raise RuntimeError(
            "No valid product names or selling prices found. "
            "No files saved."
        )

    # Preserve every RAW row, including repeated products and
    # records that may require validation in the central pipeline.
    invalid_rows = int((~(valid_names & valid_prices)).sum())

    if invalid_rows:
        print(
            f"WARNING: {invalid_rows} rows have missing names "
            "or invalid prices. RAW values will be preserved."
        )

    # --------------------------------------------------
    # CHECK COLLECTION COMPLETENESS
    # --------------------------------------------------

    expected_terms = set(SEARCH_TERMS)
    attempted_terms = set(df_summary["searched_item"])

    all_terms_attempted = (
        len(df_summary) == len(SEARCH_TERMS)
        and attempted_terms == expected_terms
    )

    all_searches_complete = (
        df_summary["status"].eq("complete").all()
    )

    all_searches_have_results = (
        df_summary["rows_extracted"].gt(0).all()
    )

    collection_complete = bool(
        all_terms_attempted
        and all_searches_complete
        and all_searches_have_results
    )

    incomplete_searches = df_summary[
        (df_summary["status"] != "complete")
        | (df_summary["rows_extracted"] <= 0)
    ]

    if not collection_complete:
        print(
            "\nWARNING: Partial collection detected. "
            "Available RAW data will still be saved."
        )

        if not incomplete_searches.empty:
            display(
                incomplete_searches[
                    [
                        "searched_item",
                        "status",
                        "products_loaded",
                        "rows_extracted",
                    ]
                ]
            )

        if not all_terms_attempted:
            print(
                "WARNING: Not all expected search terms "
                "were attempted."
            )

    # --------------------------------------------------
    # PROTECT HISTORICAL SNAPSHOTS
    # --------------------------------------------------

    if RAW_HISTORY_PATH.exists():
        raise FileExistsError(
            "A RAW historical snapshot already exists "
            f"for {RUN_DATE}: {RAW_HISTORY_PATH}. "
            "No files were modified."
        )

    # --------------------------------------------------
    # PREPARE CSV FILES
    # --------------------------------------------------

    def prepare_raw_temp(destination):
        destination.parent.mkdir(
            parents=True,
            exist_ok=True
        )

        fd, temp_name = tempfile.mkstemp(
            prefix=".ahorramas_",
            suffix=".tmp",
            dir=destination.parent
        )

        try:
            with os.fdopen(
                fd,
                "w",
                encoding="utf-8-sig",
                newline=""
            ) as file:
                df_ahorramas.to_csv(
                    file,
                    index=False
                )

            return Path(temp_name)

        except Exception:
            Path(temp_name).unlink(missing_ok=True)
            raise

    history_temp = None
    current_temp = None

    try:
        history_temp = prepare_raw_temp(RAW_HISTORY_PATH)
        current_temp = prepare_raw_temp(RAW_PATH)

        # Save history first, without overwriting an existing date.
        with history_temp.open("rb") as source:
            with RAW_HISTORY_PATH.open("xb") as target:
                while True:
                    chunk = source.read(1024 * 1024)
                    if not chunk:
                        break
                    target.write(chunk)

        print("Saved historical RAW:", RAW_HISTORY_PATH)

        # Update the latest RAW only after saving history.
        os.replace(current_temp, RAW_PATH)

        print("Saved current RAW:", RAW_PATH)

    finally:
        if history_temp is not None:
            history_temp.unlink(missing_ok=True)

        if current_temp is not None:
            current_temp.unlink(missing_ok=True)

    # --------------------------------------------------
    # COLLECTION SUMMARY
    # --------------------------------------------------

    print("\nCOLLECTION SUMMARY")
    print("Date:", RUN_DATE)
    print(
        "Status:",
        "complete" if collection_complete else "partial"
    )
    print("RAW rows:", len(df_ahorramas))
    print(
        "Distinct product IDs:",
        df_ahorramas["product_id"].nunique()
    )
    print("Searches attempted:", len(df_summary))
    print("Expected searches:", len(SEARCH_TERMS))
    print("Rows requiring validation:", invalid_rows)

    print("\nSaved outputs:")
    print(RAW_PATH)
    print(RAW_HISTORY_PATH)


TEST MODE: no files will be saved.
